In [ ]:
import pandas as pd

# Loading the data
ska_state_data_path = '/Users/fardeenb/Documents/Projects/cs3891-network-project-test/data/raw_data/ska_state.csv'
df = pd.read_csv(ska_state_data_path, delimiter=',')

# Filter data by state levels
state_data = df[df['region_code'].str.len() == 2]

# Columns of interest
columns_of_interest = [
    'all_providers', 
    'all_primary_care_providers', 
    'all_physicians', 
    'all_primary_care_physicians', 
    'all_nurse_practitioners', 
    'all_primary_care_nurse_practitioners', 
    'all_physician_assistants', 
    'all_primary_care_physician_assistants'
]

# Add the year period column for aggregation (grouping by 2011-2013)
df['period'] = df['period'].apply(lambda x: '2011-2013' if x in ['2011', '2012', '2013'] else x)

# Group by state and period (2011-2013) and sum up the provider types
aggregated_data = df.groupby(['region', 'region_code', 'period'])[columns_of_interest].sum().reset_index()

# Create the 'all_provider_types' column by summing all the provider types
aggregated_data['all_provider_types'] = aggregated_data[columns_of_interest].sum(axis=1)

# Optional: Format the 'all_provider_types' as a string
# aggregated_data['all_provider_types'] = aggregated_data[columns_of_interest].apply(lambda row: '+'.join([str(int(val)) for val in row]), axis=1)

# Select only the required columns
aggregated_data = aggregated_data[['region', 'region_code', 'period', 'all_provider_types']]

# Display the aggregated data
print(aggregated_data)

# Save to CSV for further processing
aggregated_data.to_csv('aggregated_state_data.csv', index=False)


        region region_code  period  all_provider_types
0      Alabama          AL    2011               27480
1      Alabama          AL    2012               28123
2      Alabama          AL    2013               27098
3       Alaska          AK    2011                5816
4       Alaska          AK    2012                5954
..         ...         ...     ...                 ...
145  Wisconsin          WI    2012               41171
146  Wisconsin          WI    2013               39217
147    Wyoming          WY    2011                3750
148    Wyoming          WY    2012                3928
149    Wyoming          WY    2013                3870

[150 rows x 4 columns]


In [2]:
from scipy.stats import skew, kurtosis
import pprint

# Summary Statistics containers
means = {}
std_devs = {}
minimums = {}
q1s = {}
medians = {}
q3s = {}
maximums = {}
variances = {}
ranges = {}
sum_of_squares = {}
coefficients_of_variation = {}
skewnesses = {}
kurtoses = {}



In [3]:

col_of_interest = ['all_provider_types']

# Calculate summary statistics for each column
for column in col_of_interest:
    data = aggregated_data[column]
    
    # Central Tendency & Spread
    means[column] = round(float(data.mean()), 2)
    std_devs[column] = round(float(data.std()), 2)
    variances[column] = round(float(data.var()), 2)
    ranges[column] = round(float(data.max() - data.min()), 2)
    sum_of_squares[column] = round(float((data ** 2).sum()), 2)
    minimums[column] = round(float(aggregated_data[column].min()), 2)
    maximums[column] = round(float(aggregated_data[column].max()), 2)
    
    # IQR Q1, Q2 (median), Q3
    q1s[column] = round(float(aggregated_data[column].quantile(0.25)), 2)
    medians[column] = round(float(aggregated_data[column].median()), 2)
    q3s[column] = round(float(aggregated_data[column].quantile(0.75)), 2)

    # Coefficient of Variation
    coefficients_of_variation[column] = round(float(std_devs[column] / means[column] * 100), 2)
    
    # Skewness and Kurtosis
    skewnesses[column] = round(float(skew(data)), 2)
    kurtoses[column] = round(float(kurtosis(data)), 2)

# Store summary statistics in a dictionary
summary_statistics = {
    "Means": means,
    "Standard Deviations": std_devs,
    "Minimums": minimums,
    "Variances": variances,
    "25th Percentiles (Q1)": q1s,
    "Medians (50th Percentiles)": medians,
    "75th Percentiles (Q3)": q3s,
    "Maximums": maximums,
    "Ranges": ranges,
    "Sum of Squares": sum_of_squares,
    "Coefficients of Variation (%)": coefficients_of_variation,
    "Skewness": skewnesses,
    "Kurtosis": kurtoses
}

# Print out the summary statistics
pprint.pprint(summary_statistics, width=80, sort_dicts=False)


{'Means': {'all_provider_types': 41308.09},
 'Standard Deviations': {'all_provider_types': 42824.94},
 'Minimums': {'all_provider_types': 3750.0},
 'Variances': {'all_provider_types': 1833975450.05},
 '25th Percentiles (Q1)': {'all_provider_types': 12769.0},
 'Medians (50th Percentiles)': {'all_provider_types': 29212.5},
 '75th Percentiles (Q3)': {'all_provider_types': 52571.5},
 'Maximums': {'all_provider_types': 226344.0},
 'Ranges': {'all_provider_types': 222594.0},
 'Sum of Squares': {'all_provider_types': 529216045667.0},
 'Coefficients of Variation (%)': {'all_provider_types': 103.67},
 'Skewness': {'all_provider_types': 2.17},
 'Kurtosis': {'all_provider_types': 5.25}}


In [4]:
import plotly.graph_objects as go
import pandas as pd

# Convert summary_statistics dictionary to a DataFrame
summary_df = pd.DataFrame(summary_statistics)

# Transpose the DataFrame to switch rows and columns
transposed_df = summary_df.transpose()

# Create the plotly table for summary statistics
fig = go.Figure(data=[go.Table(
    header=dict(values=["Statistics", *transposed_df.columns], fill_color='paleturquoise', align='left'),
    cells=dict(values=[list(transposed_df.index)] + [transposed_df[col].values for col in transposed_df.columns],
               fill_color='lavender', align='left')
)])

# Update layout for a cleaner look
fig.update_layout(
    title="SS for Aggregate Provider Types",
    title_x=0.5,
    width=450,  # Set a fixed width
    height=390,  # Set a fixed height
    margin=dict(l=20, r=20, t=40, b=20)
)

fig.show()
